# 04 - Demographics by Neighborhood

Pulls ACS 5-year 2023 demographic data (race/ethnicity and median income) from the Census API at the tract level for SF County. Merges with a tract-to-neighborhood crosswalk, aggregates to the neighborhood level, and computes percent shares by group. Exports `demographics_by_neighs.parquet` and `demographics_sf_city.parquet` for citywide comparisons.

<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/processing_notebooks/02d_processing_nbhd_scatterplot_demographics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# SF neighborhood geometry

nbhd_gdf = gpd.read_file('../../data/processed/polygons/sf_neighborhoods.geojson')

# Demographic Data by Neighborhood

In [3]:
from shapely import wkt

# # Read in tract to neighborhood geometry file
neighborhoods = pd.read_csv('../../data/raw/tract_to_neighborhood_geom.csv')

# Convert WKT geometry column to shapely geometry and make GeoDataFrame
neighborhoods["geometry"] = neighborhoods["the_geom"].apply(wkt.loads)
neighborhoods = gpd.GeoDataFrame(neighborhoods, geometry="geometry")

In [ ]:
neighborhoods = neighborhoods[[
    "geoid",
    "neighborhoods_analysis_boundaries",
    "geometry"
]].rename(columns={
    "geoid": "tract",
    "neighborhoods_analysis_boundaries": "neighborhood"
})

In [5]:
neighborhoods.head()

,tract,neighborhood,geometry
0,6075980900,Bayview Hunters Point,"MULTIPOLYGON (((-122.37276 37.74551, -122.3727..."
1,6075980600,Bayview Hunters Point,"MULTIPOLYGON (((-122.36519 37.73373, -122.3665..."
2,6075980501,McLaren Park,"MULTIPOLYGON (((-122.40667 37.71921, -122.4069..."
3,6075980401,The Farallones,"MULTIPOLYGON (((-123.0036 37.69325, -123.00409..."
4,6075061200,Bayview Hunters Point,"MULTIPOLYGON (((-122.38528 37.74024, -122.3858..."


In [6]:
!pip install census
from census import Census

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('CENSUS_API_KEY')
c = Census(key=api_key)

In [7]:
race_vars = {
    "B03002_001E": "population",
    "B03002_003E": "white",           # Non-Hispanic White
    "B03002_004E": "black",           # Non-Hispanic Black
    "B03002_005E": "aian",            # Non-Hispanic American Indian/Alaska Native
    "B03002_006E": "asian",           # Non-Hispanic Asian
    "B03002_007E": "nhpi",            # Non-Hispanic Native Hawaiian/Pacific Islander
    "B03002_008E": "other",           # Non-Hispanic Some Other Race
    "B03002_012E": "latina_o",        # Hispanic or Latino
}

income_vars = {
    "B19013_001E": "median_income"
}

all_vars = {**race_vars, **income_vars}

# Pulling ACS 5-year 2023 data
acs = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:075'},
        year=2023
    )
).rename(columns=all_vars)

acs['tract'] = '06075' + acs['tract']

acs_city = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'place:67000', 'in': 'state:06'},
        year=2023
    )
).rename(columns=all_vars)

In [8]:
# Replace Census null codes with NaN
acs['median_income'] = acs['median_income'].replace(-666666666, pd.NA)

In [9]:
import numpy as np

acs['tract'] = acs['tract'].astype(str).str.lstrip('0')
neighborhoods['tract'] = neighborhoods['tract'].astype(str)
acs['tract'] = acs['tract'].astype(str)

merged = neighborhoods.merge(acs, on='tract', how='inner')

agg_cols = {
    'population': 'sum',
    'white': 'sum',
    'black': 'sum',
    'aian': 'sum',
    'asian': 'sum',
    'nhpi': 'sum',
    'other': 'sum',
    'latina_o': 'sum',
}

final = merged.groupby('neighborhood').agg(agg_cols).reset_index()

def weighted_income(group):
    valid = group.dropna(subset=['median_income'])
    if valid.empty:
        return pd.NA
    return np.average(valid['median_income'], weights=valid['population'])

income = merged.groupby('neighborhood').apply(weighted_income).reset_index()
income.columns = ['neighborhood', 'median_income']

final = final.merge(income, on='neighborhood')

/var/folders/y0/kbzg62z14fl9m2k_wblyqpnc0000gp/T/ipykernel_51241/2660388922.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  income = merged.groupby('neighborhood').apply(weighted_income).reset_index()


In [10]:
# Percentages
# combining Native American into other because all the values are less than 1%
# combining NHPI with Asian because all the values are less than .02%
final['other'] = final['other'] + final['aian']
final = final.drop(columns='aian')
final['asian'] = final['asian'] + final['nhpi']
final = final.drop(columns='nhpi')

final['pct_other']   = final['other']   / final['population']
final['pct_white']   = final['white']   / final['population']
final['pct_black']   = final['black']   / final['population']
final['pct_asian']   = final['asian']   / final['population']
final['pct_latina_o']= final['latina_o']/ final['population']

# Filter out very small nbhds
final_df = final[final['population'] >= 200]

final_df

,neighborhood,population,white,black,asian,other,latina_o,median_income,pct_other,pct_white,pct_black,pct_asian,pct_latina_o
0,Bayview Hunters Point,39816.0,3055.0,9295.0,15905.0,31.0,9733.0,99166.872614,0.000779,0.076728,0.233449,0.399463,0.244449
1,Bernal Heights,24725.0,10202.0,754.0,3419.0,116.0,8171.0,162466.201173,0.004692,0.412619,0.030495,0.138281,0.330475
2,Castro/Upper Market,22024.0,14597.0,552.0,2998.0,173.0,2178.0,203862.273066,0.007855,0.662777,0.025064,0.136124,0.098892
3,Chinatown,12644.0,1239.0,178.0,10282.0,16.0,534.0,37973.447722,0.001265,0.097991,0.014078,0.813192,0.042233
4,Excelsior,37915.0,5776.0,582.0,18710.0,408.0,11607.0,109720.738362,0.010761,0.152341,0.015350,0.493472,0.306132
5,Financial District/South Beach,24519.0,9394.0,491.0,11296.0,31.0,2282.0,190544.602145,0.001264,0.383131,0.020025,0.460704,0.093071
6,Glen Park,8458.0,4630.0,528.0,1260.0,84.0,953.0,179044.869591,0.009931,0.547411,0.062426,0.148971,0.112674
8,Haight Ashbury,17780.0,12062.0,472.0,2254.0,111.0,1287.0,201787.62396,0.006243,0.678403,0.026547,0.126772,0.072385
9,Hayes Valley,18240.0,9421.0,1335.0,3634.0,545.0,2293.0,133318.436294,0.029879,0.516502,0.073191,0.199232,0.125713
10,Inner Richmond,20261.0,9384.0,284.0,7211.0,130.0,1942.0,170174.100686,0.006416,0.463156,0.014017,0.355905,0.095849


In [11]:
acs_city.to_parquet('../../data/processed/demographics_sf_city.parquet', index=False)

In [12]:
print(acs_city.columns.tolist())

['population', 'white', 'black', 'aian', 'asian', 'nhpi', 'other', 'latina_o', 'median_income', 'state', 'place']


In [13]:
final_df.to_parquet('../../data/processed/demographics_by_neighs.parquet')
acs

,population,white,black,aian,asian,nhpi,other,latina_o,median_income,state,county,tract
0,2004.0,770.0,132.0,52.0,608.0,0.0,47.0,336.0,101974.0,06,075,6075010101
1,1795.0,554.0,322.0,0.0,766.0,16.0,0.0,93.0,108417.0,06,075,6075010102
2,2608.0,1762.0,134.0,0.0,206.0,0.0,0.0,311.0,160221.0,06,075,6075010201
3,1761.0,1179.0,58.0,0.0,284.0,27.0,38.0,55.0,206484.0,06,075,6075010202
4,3791.0,2258.0,0.0,0.0,949.0,0.0,28.0,453.0,158973.0,06,075,6075010300
...,...,...,...,...,...,...,...,...,...,...,...,...
239,155.0,0.0,0.0,0.0,155.0,0.0,0.0,0.0,13569.0,06,075,6075980501
240,1290.0,242.0,288.0,0.0,463.0,0.0,1.0,194.0,142321.0,06,075,6075980600
241,322.0,32.0,27.0,0.0,214.0,0.0,0.0,20.0,53800.0,06,075,6075980900
242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,06,075,6075990100
